In [1]:
%load_ext autoreload

In [2]:
%autoreload 2
import torch

In [3]:
code_batch = torch.randn(10, 16)

n_batch, n_latent = code_batch.shape
device = code_batch.device

In [4]:
# Thresholds from Kennel et al. 1992
rtol = 20.0
atol = 2.0

# Lower triangular mask: for dimension d, keep only first d coordinates
tri_mask = torch.tril(
    torch.ones(n_latent, n_latent, dtype=torch.float32, device=device)
)

In [5]:
# batch_masked: (n_latent, batch_size, n_latent)
batch_masked = tri_mask[:, None, :] * code_batch[None, :, :]

In [6]:
# Pairwise squared distances for each incremental dimension set
X_sq = (batch_masked * batch_masked).sum(dim=2, keepdim=True)
X_sq.shape

torch.Size([16, 10, 1])

In [7]:
pdist_vector = (
        X_sq
        + X_sq.transpose(1, 2)
        - 2 * torch.bmm(batch_masked, batch_masked.transpose(1, 2))
    )
all_dists = pdist_vector  # (n_latent, batch_size, batch_size)
# Avoid zero distances
all_dists = torch.clamp(all_dists, min=1e-14)

**Computing the characeristic size of the attractor for the first $m$ latent coordinates**

From the paper, the characteristic size of the attractor when using the first $m$ latent coordinates is given by:


\begin{aligned}
\mathcal{R}^2_m &= \frac{1}{m B} \sum_{b=1}^B \sum_{i=1}^m (h_{bi} - \bar{h}_i)^2 \\
&= \frac{1}{m} \sum_{i=1}^m \left( \frac{1}{B} \sum_{b=1}^B (h_{bi} - \bar{h}_i)^2 \right) \\
&= \frac{1}{m} \sum_{i=1}^m \sigma_i^2
\end{aligned}
where $\sigma_i^2$ is the variance of the $i$-th coordinate over the batch.





In [17]:
stds = torch.std(batch_masked, dim=1, keepdim=True)  # (n_latent, 1, n_latent)
all_ra = torch.sqrt(
        (1.0 / torch.arange(1, 1 + n_latent, dtype=torch.float32, device=device))
        * (stds ** 2).sum(dim=2).squeeze(1)
    )  # (n_latent,)

In [8]:
# Find k+1 nearest neighbors (smallest distances)
k = 1
_, inds = torch.topk(-all_dists, k + 1, dim=-1)

In [9]:
neighbor_dists_d = torch.gather(all_dists, 2, inds)

In [10]:
# Gather distances at dimension d+1 using neighbors found at dimension d
neighbor_new_dists = torch.gather(all_dists[1:], 2, inds[:-1])
neighbor_new_dists.shape

torch.Size([15, 10, 2])

In [15]:
scaled_dist = torch.sqrt(
        (neighbor_new_dists - neighbor_dists_d[:-1])
        / neighbor_dists_d[:-1]
    )

In [18]:
# Kennel condition #1: distance ratio exceeds threshold
is_false_change = scaled_dist > rtol
# Kennel condition #2: absolute distance exceeds threshold
is_large_jump = neighbor_new_dists > atol * all_ra[:-1, None, None]
# NOTE: in the paper, it is written that one compares $\tilde{D}^2_{abm}$ to $A_{tol} R_m$. However, in the Gilpin's fnn
# repo, and in the original Kennel et al. paper, they compare $\tilde{D}'^2_{abm}$ to $A_{tol} R_m$. I believe that this
# is again a typo in the paper, and that $\tilde{D}'^2_{abm}$ should be used in its definition.

is_false_neighbor = torch.logical_or(is_false_change, is_large_jump)
total_false_neighbors = is_false_neighbor.to(torch.int32)[..., 1:(k + 1)]

In [27]:
reg_weights = 1 - total_false_neighbors.to(torch.float64).mean(dim=(1, 2))
reg_weights = torch.nn.functional.pad(reg_weights, (1, 0))  # pad zero for dim 0

In [28]:
reg_weights

tensor([0.0000, 0.8000, 0.5000, 0.3000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       dtype=torch.float64)

In [29]:
code_batch.shape

torch.Size([10, 16])